<img src="https://10botics.com/logo_jnb.png" width="300"/>

# Raspberry Pi Button Testing Tutorial

## Learning Objectives

In this lesson, you will learn:

1. Wire up a 100mm arcade button to Raspberry Pi
2. Use the `gpiozero` library to detect button presses
3. Understand button states and transitions
4. Handle button debouncing and error conditions
5. Create interactive button detection programs
6. Clean up gpiozero resources properly

## Hardware Setup

Before we start coding, you need to wire up your button:

<img src="./images/microswitch.png" width=600/>

### Wiring Diagram
```
Raspberry Pi    Arcade Button
GND     -----> COM (Common)
GPIO 6  -----> NO (Normally Open)
```

### Steps:
1. Connect the button's **COM** (common) terminal to Raspberry Pi **GND**
2. Connect the button's **NO** (normally open) terminal to Raspberry Pi **GPIO 6**
3. Make sure your Raspberry Pi is powered on

**Note:** We're using BCM pin numbering (GPIO 6), not physical pin numbers.

## Step 1: Verify Pi 5 Libraries

**Objective:** Verify the correct GPIO libraries for the Pi 5 are installed.

**Instructions:** On the Pi 5, we must use gpiozero and its lgpio backend. These are usually pre-installed, but we will run this command in the Terminal to be sure.

In [ ]:
!sudo apt update
!sudo apt install python3-gpiozero python3-lgpio -y

## Step 2: Import Required Libraries

**Objective:** Import the necessary libraries for button handling

**Instructions:** Import the Button object from the gpiozero library and the time library. We no longer need RPi.GPIO.

In [ ]:
# Import the Button object from the gpiozero library
from gpiozero import Button
# Import the time library for later use
import time

print("✅ gpiozero and time libraries imported successfully")

## Step 3: Setup GPIO Configuration

**Objective:** Configure GPIO for button input. 

**Instructions:** Complete the code below to create a Button object. This single line will configure the pin.

**About the parameters:**
- `button = Button(6, ...)`: This creates the object. The 6 automatically signifies BCM pin 6.
- `pull_up=True`: This configures the pin as an input and enables the internal pull-up resistor.
- When the button is not pressed, the pin is HIGH, and button.is_pressed will be False.
- When the button is pressed (connecting pin to GND), the pin goes LOW, and button.is_pressed will be True.

In [ ]:
# Create a Button object for BCM pin 6 with a pull-up resistor
button = Button(6, pull_up=True)

print("✅ GPIO configured for button input (using gpiozero)")
print(f"Button pin: GPIO {button.pin.number} (BCM)")
print(f"Pull-up resistor: Enabled")

# (Note: If you run this cell again, you will get a "pin in use" error. This is normal. Just "Restart Kernel" from the menu to clear the old object.)

## Exercise 4: Understand Button States

**Objective:** Read the button's current logical state and raw electrical state.

**Instructions:** Use the button object created in Exercise 3 to read its properties.

**About button states:**
- `button.pin.state`: This property gets the raw electrical value of the pin. 1 = HIGH, 0 = LOW. This is the direct equivalent of the old GPIO.input(6).
- `button.is_pressed`: This property gets the logical state. Because we set pull_up=True, gpiozero knows the button is "pressed" when the pin state is LOW (0). It automatically returns True (if pressed) or False (if not pressed). This replaces the entire if button_state == GPIO.LOW: block.

In [ ]:
print("Current button state:")

# 1. Get the raw pin value (1 for HIGH, 0 for LOW)
# This is the equivalent of RPi.GPIO's: button_state = GPIO.input(6)
raw_pin_value = ????
print(f"Raw pin value: {raw_pin_value}")


# 2. Get the logical "pressed" state (True or False)
# This replaces the old if/else block that checked for GPIO.LOW
button_pressed = ????
print(f"Button pressed: {button_pressed}")

### Answer

In [ ]:
print("Current button state:")

# 1. Get the raw pin value (1 for HIGH, 0 for LOW)
# This is the equivalent of RPi.GPIO's: button_state = GPIO.input(6)
raw_pin_value = button.pin.state
print(f"Raw pin value: {raw_pin_value}")


# 2. Get the logical "pressed" state (True or False)
# This replaces the old if/else block that checked for GPIO.LOW
button_pressed = button.is_pressed   
print(f"Button pressed: {button_pressed}")

## Exercise 5: Simple Button Press Detection

**Objective:** Create a loop that detects button presses and releases

**Instructions:** Complete the code below to detect button presses and releases for 10 seconds

**About the code:**
- We track the previous state to detect changes
- When `current_state_pressed` is `True` and `last_state_pressed` is `False`, the button was just pressed
- When `current_state_pressed` is `False` and `last_state_pressed` is `True`, the button was just released

In [ ]:
import time

print("Starting 10-second button detection loop (Polling Method)...")
print("Try pressing and releasing the button.")

press_count = 0
# Get the initial state (False if not pressed, True if pressed)
last_state_pressed = button.is_pressed

# Run for 10 seconds to test
start_time = time.time()

while time.time() - start_time < 10:
    # Get the current logical state
    current_state_pressed = ????
    
    # Detect a change in state
    if current_state_pressed != last_state_pressed:
        
        # Detect button press (transition from False to True)
        if current_state_pressed:
            press_count += 1
            print(f"🔘 Button PRESSED! (Press #{press_count})")
        
        # Detect button release (transition from True to False)
        else:
            print(f"   Button released")
        
    # Update the last_state for the next loop iteration
    last_state_pressed = current_state_pressed
    
    # 10ms polling interval to prevent 100% CPU usage
    time.sleep(0.01) 

print(f"\n✅ Test completed! Total presses detected: {press_count}")

### Answer

In [ ]:
import time

print("Starting 10-second button detection loop (Polling Method)...")
print("Try pressing and releasing the button.")

press_count = 0
# Get the initial state (False if not pressed, True if pressed)
last_state_pressed = button.is_pressed

# Run for 10 seconds to test
start_time = time.time()

while time.time() - start_time < 10:
    # Get the current logical state
    current_state_pressed = button.is_pressed
    
    # Detect a change in state
    if current_state_pressed != last_state_pressed:
        
        # Detect button press (transition from False to True)
        if current_state_pressed:
            press_count += 1
            print(f"🔘 Button PRESSED! (Press #{press_count})")
        
        # Detect button release (transition from True to False)
        else:
            print(f"   Button released")
        
    # Update the last_state for the next loop iteration
    last_state_pressed = current_state_pressed
    
    # 10ms polling interval to prevent 100% CPU usage
    time.sleep(0.01) 

print(f"\n✅ Test completed! Total presses detected: {press_count}")

## Exercise 7: Interactive Button Counter

**Objective:** Create an interactive program that counts button presses until interrupted

**Instructions:** Complete the code below to create an interactive button counter

**About the code:**
- This runs forever until you press Ctrl+C to stop it
- We use the button object (Pin 6) that we already created.
- The logic is the same as Exercise 5, but in a while True loop.
- The logic is the same as the previous exercise, but without a time limit.
- We use try...except KeyboardInterrupt to catch the "Stop" command and end the loop gracefully.

In [ ]:
import time

print("Interactive Button Counter (using GPIO 6)")
print("=======================================")
print(f"Listening on GPIO {button.pin.number}...")
print("Press the button multiple times to test...")
print("Press the Jupyter 'Stop' (■) button to stop")
print()

press_count = 0
# Get initial state for pin 6
last_state_pressed = button.is_pressed

while True:
    # Complete this line to get current button state:
    current_state_pressed = ????
    
    # Detect button press (transition from False to True)
    if current_state_pressed and not last_state_pressed:
        press_count += 1
        print(f"🔘 Press #{press_count} detected!")
    
    # Detect button release (transition from True to False)
    elif not current_state_pressed and last_state_pressed:
        print(f"   Released")
    
    last_state_pressed = current_state_pressed
    time.sleep(0.01)


### Answer

In [ ]:
import time

print("Interactive Button Counter (using GPIO 6)")
print("=======================================")
print(f"Listening on GPIO {button.pin.number}...")
print("Press the button multiple times to test...")
print("Press the Jupyter 'Stop' (■) button to stop")
print()

press_count = 0
# Get initial state for pin 6
last_state_pressed = button.is_pressed

while True:
    # Complete this line to get current button state:
    current_state_pressed = button.is_pressed
    
    # Detect button press (transition from False to True)
    if current_state_pressed and not last_state_pressed:
        press_count += 1
        print(f"🔘 Press #{press_count} detected!")
    
    # Detect button release (transition from True to False)
    elif not current_state_pressed and last_state_pressed:
        print(f"   Released")
    
    last_state_pressed = current_state_pressed
    time.sleep(0.01)


## Exercise 8: Clean Up Resources

**Objective:** Properly clean up GPIO resources when done

**Instructions:** In gpiozero, we don't use a global GPIO.cleanup(). Instead, we .close() the specific objects we created. This is much better for Jupyter, as it prevents errors when re-running cells.

We already added .close() to our try...except block in the last exercise. Let's formally close our first button object (from Pin 6) to finish.

In [ ]:
# Complete this line to clean up the *original* button (pin 6) object:
# Your code here...
# Hint: button.close()

print("✅ GPIO 6 resources cleaned up")
print("\nTutorial completed! 🎉")

### Answer

In [ ]:
# Close the original button object from Exercise 3
button.close()
    
print("✅ GPIO 6 resources cleaned up")    
print("\nTutorial completed! 🎉")

## Challenges

### Challenge 1: Button Hold Detection

**Objective:** Detect when a button is held down for more than 2 seconds

**Instructions:** Create code to detect when the button is held for 2+ seconds

**Hints:** Track the time when the button was first pressed using `time.time()`

In [ ]:
import time

# We will use the 'button' object (GPIO 6) created in Exercise 3

print("Button Hold Detection (using GPIO 6)")
print("====================")
print(f"Listening on GPIO {button.pin.number}...")
print("Hold the button for 2+ seconds to test...")
print("Press 'Stop' (■) to exit.")
print()

# Write your code here to monitor all buttons
# Check which buttons are pressed and display combinations

# Variables you might need:
# last_states_pressed = [b.is_pressed for b in buttons]

# Your code here...

# (Remember to wrap your loop in a try...except...finally block!)

### Answer

In [ ]:
import time

# We will use the 'button' object (GPIO 6) created in Exercise 3

print("Button Hold Detection (using GPIO 6)")
print("====================")
print(f"Listening on GPIO {button.pin.number}...")
print("Hold the button for 2+ seconds to test...")
print("Press 'Stop' (■) to exit.")
print()

press_start_time = None
hold_detected = False
last_state_pressed = button.is_pressed

while True:
    current_state_pressed = button.is_pressed
    current_time = time.time()
    
    # Button just pressed (False -> True)
    if current_state_pressed and not last_state_pressed:
        press_start_time = current_time
        hold_detected = False
        print("🔘 Button pressed - start timing...")
    
    # Button held down (True -> True)
    elif current_state_pressed and last_state_pressed and press_start_time:
        hold_duration = current_time - press_start_time
        if hold_duration >= 2.0 and not hold_detected:
            hold_detected = True
            print(f"⚠️  Button held for {hold_duration:.1f} seconds!")
    
    # Button released (True -> False)
    elif not current_state_pressed and last_state_pressed:
        if press_start_time:
            total_duration = current_time - press_start_time
            print(f"   Button released after {total_duration:.1f} seconds")
        press_start_time = None
    
    last_state_pressed = current_state_pressed
    time.sleep(0.01)


### Challenge 2: Multiple Button Detection

**Objective:** Detect multiple buttons pressed simultaneously

**Instructions:** Setup multiple Button objects and monitor all of them for combinations.

**Hints:** Create a list of Button objects and check their is_pressed states in a loop.

In [ ]:
# Setup multiple button pins (GPIO 5, 2, 3, 4)
button_pins = [5, 2, 3, 4]
buttons = []
try:
    # Create a list of Button objects
    buttons = [Button(pin, pull_up=True) for pin in button_pins]
    print(f"Monitoring buttons on BCM pins: {button_pins}")
except Exception as e:
    print(f"Error creating buttons: {e}")
    print("Please 'Restart Kernel' and try again.")
# -------------------------

print("Multiple Button Detection")
print("=======================")
print("Press any combination of buttons...")
print("Press 'Stop' (■) to exit.")
print()

# Write your code here to monitor all buttons
# Check which buttons are pressed and display combinations

# Variables you might need:
# last_states_pressed = [b.is_pressed for b in buttons]

# Your code here...

# (Remember to wrap your loop in a try...except...finally block!)

### Answer

In [ ]:
import time

# Setup multiple button pins (GPIO 5, 2, 3, 4)
button_pins = [5, 2, 3, 4]
buttons = []
try:
    # Create a list of Button objects
    buttons = [Button(pin, pull_up=True) for pin in button_pins]
    print(f"Monitoring buttons on BCM pins: {button_pins}")
except Exception as e:
    print(f"Error creating buttons: {e}")
    print("Please 'Restart Kernel' and try again.")
# -------------------------

print("Multiple Button Detection")
print("=======================")
print("Press any combination of buttons...")
print("Press 'Stop' (■) to exit.")
print()

try:
    # Get initial states
    last_states_pressed = [b.is_pressed for b in buttons]

    while True:
        # Get current states
        current_states_pressed = [b.is_pressed for b in buttons]
        
        # Check each button for state changes
        for i, (current, last) in enumerate(zip(current_states_pressed, last_states_pressed)):
            pin_num = button_pins[i]
            if current and not last: # Just pressed (False -> True)
                print(f"🔘 Button on GPIO {pin_num} pressed")
            elif not current and last: # Just released (True -> False)
                print(f"   Button on GPIO {pin_num} released")
        
        # Show current combination
        pressed_pins = [button_pins[i] for i, pressed in enumerate(current_states_pressed) if pressed]
        if pressed_pins:
            print(f"   Currently pressed: {pressed_pins}")
        
        last_states_pressed = current_states_pressed.copy()
        time.sleep(0.01)

except KeyboardInterrupt:
    print("\nChallenge 2 stopped.")
except NameError:
    print("\nError: 'buttons' list not created. Please re-run the cell.")
finally:
    # Clean up all button objects in the list
    if buttons:
        for b in buttons:
            b.close()
        print(f"GPIO {button_pins} resources cleaned up.")

## Congratulations! 🎉

You have successfully completed the Raspberry Pi Button Testing Tutorial!

### What You've Learned:
- How to wire up arcade buttons to Raspberry Pi
- How to use the `gpiozero` library for button input
- How to detect button presses and releases
- How to handle button debouncing and state management
- How to create interactive button programs
- How to properly clean up GPIO resources

### Next Steps:
- Try the challenges above
- Experiment with different button configurations
- Move on to LED control with RPi.GPIO
- Build the full LED racing game!

### Troubleshooting Tips:
- If buttons aren't detected, check your wiring
- Make sure you're running on a Raspberry Pi
- Verify the button is working with a multimeter
- Check GPIO permissions if you get errors
- **RPi.GPIO Advantage:** Use `GPIO.cleanup()` to reset all pins

<hr/>

## Congratulation! You have finished this chapter.

This jupyter notebook is created by 10Botics. <br>
For permission to use in school, please contact info@10botics.com <br>
All rights reserved. 2025.